In [1]:
import pandas as pd

file_path = '/content/Titanic-Dataset.csv'
try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded data from {file_path}")
except FileNotFoundError:
    print(f"Error: The file {file_path} was not found. Please ensure the dataset file is in the correct directory.")
    df = pd.DataFrame()
except Exception as e:
    print(f"An error occurred while loading the CSV file: {e}")
    df = pd.DataFrame()


Successfully loaded data from /content/Titanic-Dataset.csv


In [2]:
if not df.empty:
    print("\n--- Dataset Head ---")
    display(df.head())

    print("\n--- Dataset Info ---")
    df.info()
else:
    print("DataFrame is empty. Please check the previous cell for errors during loading.")



--- Dataset Head ---


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


## 2. EDA  - Text Analysis of the 'Name' Column


In [3]:
print(f"Number of unique names: {df['Name'].nunique()}")
print(f"First 10 names:\n{df['Name'].head(10).tolist()}")


Number of unique names: 891
First 10 names:
['Braund, Mr. Owen Harris', 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)', 'Heikkinen, Miss. Laina', 'Futrelle, Mrs. Jacques Heath (Lily May Peel)', 'Allen, Mr. William Henry', 'Moran, Mr. James', 'McCarthy, Mr. Timothy J', 'Palsson, Master. Gosta Leonard', 'Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)', 'Nasser, Mrs. Nicholas (Adele Achem)']


### 3. Data Preprocessing


In [4]:
df['Title'] = df['Name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())
rare_titles = df['Title'].value_counts()[df['Title'].value_counts() < 10].index
df['Title'] = df['Title'].replace(rare_titles, 'Rare')

print("Distribution of Titles after consolidation:")
display(df['Title'].value_counts())

Distribution of Titles after consolidation:


,count
Title,
Mr,517
Miss,182
Mrs,125
Master,40
Rare,27


### 3.1 Cleaning the Corpus

In [5]:
import re

def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-z ]', '', text)  # Remove non-alphabetic characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

df['Cleaned_Name'] = df['Name'].apply(clean_text)

print("First 10 original names vs. cleaned names:")
for i in range(10):
    print(f"Original: {df['Name'].iloc[i]} -> Cleaned: {df['Cleaned_Name'].iloc[i]}")

First 10 original names vs. cleaned names:
Original: Braund, Mr. Owen Harris -> Cleaned: braund mr owen harris
Original: Cumings, Mrs. John Bradley (Florence Briggs Thayer) -> Cleaned: cumings mrs john bradley florence briggs thayer
Original: Heikkinen, Miss. Laina -> Cleaned: heikkinen miss laina
Original: Futrelle, Mrs. Jacques Heath (Lily May Peel) -> Cleaned: futrelle mrs jacques heath lily may peel
Original: Allen, Mr. William Henry -> Cleaned: allen mr william henry
Original: Moran, Mr. James -> Cleaned: moran mr james
Original: McCarthy, Mr. Timothy J -> Cleaned: mccarthy mr timothy j
Original: Palsson, Master. Gosta Leonard -> Cleaned: palsson master gosta leonard
Original: Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg) -> Cleaned: johnson mrs oscar w elisabeth vilhelmina berg
Original: Nasser, Mrs. Nicholas (Adele Achem) -> Cleaned: nasser mrs nicholas adele achem


### 3.2 Stemming



In [ ]:
import nltk
from nltk.stem.porter import PorterStemmer
nltk.download('punkt')

stemmer = PorterStemmer()

def stem_text(text):
    return ' '.join([stemmer.stem(word) for word in text.split()])

df['Stemmed_Name'] = df['Cleaned_Name'].apply(stem_text)

print("First 10 cleaned names vs. stemmed names:")
for i in range(10):
    print(f"Cleaned: {df['Cleaned_Name'].iloc[i]} -> Stemmed: {df['Stemmed_Name'].iloc[i]}")

## 4. Tokens visualization



In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

all_stemmed_names = ' '.join(df['Stemmed_Name'].dropna().tolist())
words = all_stemmed_names.split()
word_counts = Counter(words)

most_common_words = word_counts.most_common(20)

print("20 Most Common Stemmed Words:")
for word, count in most_common_words:
    print(f"{word}: {count}")

words_df = pd.DataFrame(most_common_words, columns=['Word', 'Count'])

plt.figure(figsize=(12, 6))
sns.barplot(x='Count', y='Word', data=words_df, palette='viridis')
plt.title('20 Most Common Stemmed Words in Names')
plt.xlabel('Frequency')
plt.ylabel('Word')
plt.show()

### 5.1 Tunning CountVectorizer


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
count_vectorizer = CountVectorizer(max_features=500)
X_count = count_vectorizer.fit_transform(df['Stemmed_Name'].dropna())
feature_names = count_vectorizer.get_feature_names_out()
count_df = pd.DataFrame(X_count.toarray(), columns=feature_names)

print("Shape of CountVectorizer output:", X_count.shape)
print("First 5 rows of CountVectorizer DataFrame (sample):")
display(count_df.head())


### 5.2 TF-IDF



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=500)
X_tfidf = tfidf_vectorizer.fit_transform(df['Stemmed_Name'].dropna())
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_feature_names)

print("Shape of TF-IDF Vectorizer output:", X_tfidf.shape)
print("First 5 rows of TF-IDF Vectorizer DataFrame (sample):")
display(tfidf_df.head())

### 5.3 Word Embeddings: GloVe


In [ ]:
import numpy as np

!wget -nc http://nlp.stanford.edu/data/glove.6B.zip
!unzip -n glove.6B.zip

def load_glove_model(glove_file):
    print("Loading GloVe Model...")
    glove_model = {}
    with open(glove_file, 'r', encoding='utf-8') as f:
        for line in f:
            split_line = line.split()
            word = split_line[0]
            embedding = np.array(split_line[1:], dtype=np.float32)
            glove_model[word] = embedding
    print(f"{len(glove_model)} words loaded!")
    return glove_model

glove_path = './glove.6B.50d.txt'
glove_model = load_glove_model(glove_path)

def get_avg_word_embedding(text, model, vector_size):
    words = text.split()
    embeddings = [model[word] for word in words if word in model]
    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        return np.zeros(vector_size)

vector_size = len(next(iter(glove_model.values()))) if glove_model else 50

df['GloVe_Embeddings'] = df['Stemmed_Name'].apply(lambda x: get_avg_word_embedding(x, glove_model, vector_size))

print("\nFirst 5 rows of GloVe Embeddings (average vector for each name):")
display(df['GloVe_Embeddings'].head())

## 6. Modeling


In [ ]:
from sklearn.model_selection import train_test_split

# Assuming X_count (from CountVectorizer) is our feature matrix and 'Survived' is our target
# We need to ensure that the number of samples for features and target match.
# X_count.shape is (891, 500), df.shape is (891, 16)

y = df['Survived']

# For simplicity, let's use the CountVectorizer output for now.
# We need to handle potential NaN values in 'Stemmed_Name' that were dropped during vectorization.
# Let's re-align X_count with the original DataFrame index.

# Create a boolean mask for non-null Stemmed_Name entries
stemmed_name_mask = df['Stemmed_Name'].notna()

# Filter the target variable y and the feature matrix X_count using this mask
y_filtered = y[stemmed_name_mask]
X_count_filtered = X_count[stemmed_name_mask.values] # Corrected line: Added .values to convert Series to NumPy array

# Split the data into training and testing sets
X_train_count, X_test_count, y_train, y_test = train_test_split(X_count_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered)

print(f"Shape of X_train_count: {X_train_count.shape}")
print(f"Shape of X_test_count: {X_test_count.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

### 6.1 Naive Bayes DTM (Document Term Matrix)

### 6.2 Naive Bayes TF-IDF

Now, let's train a Multinomial Naive Bayes classifier using the TF-IDF vectorized features (`X_tfidf_filtered`). This approach weighs words based on their importance across the corpus, potentially improving model performance.

In [ ]:
from sklearn.model_selection import train_test_split

# Filter X_tfidf based on the stemmed_name_mask
X_tfidf_filtered = X_tfidf[stemmed_name_mask.values]

# Split the TF-IDF data into training and testing sets
X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(X_tfidf_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered)

print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")
print(f"Shape of y_train_tfidf: {y_train_tfidf.shape}")
print(f"Shape of y_test_tfidf: {y_test_tfidf.shape}")

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize and train the Multinomial Naive Bayes model with TF-IDF features
mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train_tfidf)

# Make predictions on the test set
y_pred_tfidf = mnb_tfidf.predict(X_test_tfidf)

# Evaluate the model
print("--- Multinomial Naive Bayes (TF-IDF) Performance ---")
print(f"Accuracy: {accuracy_score(y_test_tfidf, y_pred_tfidf):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_tfidf, y_pred_tfidf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_tfidf, y_pred_tfidf))

### 6.3 XGBoost

Now, let's train an XGBoost classifier. XGBoost is known for its strong predictive performance and is a popular choice for structured and text-based data.

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize and train the XGBoost classifier with CountVectorizer features
# X_train_count and y_train are already defined from the previous CountVectorizer section
xgb_classifier = xgb.XGBClassifier(
    objective='binary:logistic', # For binary classification
    eval_metric='logloss',       # Evaluation metric
    use_label_encoder=False,     # Suppress warning for label encoder
    random_state=42
)

xgb_classifier.fit(X_train_count, y_train)

# Make predictions on the test set
y_pred_xgb = xgb_classifier.predict(X_test_count)

# Evaluate the model
print("--- XGBoost (CountVectorizer) Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

## 7. LSTM

Long Short-Term Memory (LSTM) networks are a type of recurrent neural network (RNN) capable of learning long-term dependencies. They are well-suited for sequence prediction problems like text classification.

### 7.1 Data Preparation for LSTM

Before feeding the text data into an LSTM, we need to convert the stemmed names into numerical sequences and ensure all sequences have the same length. This involves tokenization and padding.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Filter stemmed names based on the mask used for previous models
stemmed_names_for_lstm = df['Stemmed_Name'][stemmed_name_mask].tolist()
y_lstm = y_filtered.values # Use the filtered target variable

# Tokenize the words
tokenizer = Tokenizer(num_words=5000, oov_token="<unk>") # num_words can be adjusted
tokenizer.fit_on_texts(stemmed_names_for_lstm)

# Convert text to sequences of integers
sequences = tokenizer.texts_to_sequences(stemmed_names_for_lstm)

# Determine maximum sequence length for padding
max_sequence_len = max([len(x) for x in sequences])
print(f"Maximum sequence length: {max_sequence_len}")

# Pad sequences to ensure uniform input size
X_lstm = pad_sequences(sequences, maxlen=max_sequence_len, padding='post')

print(f"Shape of X_lstm (padded sequences): {X_lstm.shape}")
print(f"Shape of y_lstm: {y_lstm.shape}")

### 7.2 Building and Training the LSTM Model

Now, we can define, compile, and train our LSTM model using the prepared sequences.

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Split data into training and testing sets for LSTM
X_train_lstm, X_test_lstm, y_train_lstm, y_test_lstm = train_test_split(X_lstm, y_lstm, test_size=0.2, random_state=42, stratify=y_lstm)

# Define vocabulary size
vocab_size = len(tokenizer.word_index) + 1

# Build the LSTM model
embedding_dim = 100 # Can be tuned
model = Sequential([
    Embedding(vocab_size, embedding_dim), # Removed input_length as it's deprecated and caused the model to not build
    LSTM(128, return_sequences=False), # LSTM layer with 128 units
    Dropout(0.5), # Dropout for regularization
    Dense(64, activation='relu'), # Dense layer
    Dropout(0.5), # Another Dropout layer
    Dense(1, activation='sigmoid') # Output layer for binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print(model.summary())

# Train the model
# Early stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train_lstm, y_train_lstm,
    epochs=10, # Number of epochs can be tuned
    batch_size=32, # Batch size can be tuned
    validation_split=0.2, # Use a validation split from the training data
    callbacks=[early_stopping]
)

# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test_lstm, y_test_lstm, verbose=0)
print(f"\nLSTM Model Test Accuracy: {accuracy:.4f}")

# Make predictions for classification report and confusion matrix
y_pred_lstm_prob = model.predict(X_test_lstm)
y_pred_lstm = (y_pred_lstm_prob > 0.5).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print("\nClassification Report (LSTM):\n")
print(classification_report(y_test_lstm, y_pred_lstm))

print("\nConfusion Matrix (LSTM):\n")
print(confusion_matrix(y_test_lstm, y_pred_lstm))

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize and train the Multinomial Naive Bayes model
mnb_dtm = MultinomialNB()
mnb_dtm.fit(X_train_count, y_train)

# Make predictions on the test set
y_pred_dtm = mnb_dtm.predict(X_test_count)

# Evaluate the model
print("--- Multinomial Naive Bayes (DTM) Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dtm):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dtm))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dtm))

## 8. BERT

Bidirectional Encoder Representations from Transformers (BERT) is a powerful pre-trained language model that can be fine-tuned for various NLP tasks, including text classification. BERT understands the context of words based on all of their surroundings (bidirectional), unlike previous models that processed text in a single direction.

### 8.1 Setup and Data Preparation for BERT

We need to install the `transformers` library from Hugging Face, which provides easy access to pre-trained BERT models and tokenizers. Then, we will tokenize and encode our `Stemmed_Name` data into a format suitable for BERT.

## 9. NLP: Disaster Tweets

This section will focus on performing Natural Language Processing tasks on a dataset related to disaster tweets. We will start with Exploratory Data Analysis (EDA) to understand the dataset, followed by preprocessing, visualization, and modeling.

### 9.1 EDA

For the Disaster Tweets dataset, we'll perform EDA to understand the distribution of target variables (disaster vs. non-disaster), tweet lengths, common words, and any other relevant insights. Please ensure the 'disaster_tweets.csv' dataset is available. If it's not present, you might need to upload it or provide a link to download it.

In [ ]:
import pandas as pd

# Load the disaster tweets dataset
# Assuming the dataset is named 'disaster_tweets.csv'
# If the file path is different, please update it.
file_path_disaster = '/content/tweets.csv'
try:
    df_tweets = pd.read_csv(file_path_disaster)
    print(f"Successfully loaded data from {file_path_disaster}")
    print("\n--- Disaster Tweets Dataset Head ---")
    display(df_tweets.head())
    print("\n--- Disaster Tweets Dataset Info ---")
    df_tweets.info()
except FileNotFoundError:
    print(f"Error: The file {file_path_disaster} was not found. Please ensure the dataset file is in the correct directory.")
    df_tweets = pd.DataFrame()
except Exception as e:
    print(f"An error occurred while loading the CSV file: {e}")
    df_tweets = pd.DataFrame()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the target variable
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df_tweets, palette='viridis')
plt.title('Distribution of Target Variable (Disaster vs. Non-Disaster Tweets)')
plt.xlabel('Target (0 = Not Disaster, 1 = Disaster)')
plt.ylabel('Number of Tweets')
plt.xticks(ticks=[0, 1], labels=['Not Disaster', 'Disaster'])
plt.show()

In [ ]:
# Distribution of the 'keyword' variable
plt.figure(figsize=(12, 6))
sns.countplot(y='keyword', data=df_tweets, order=df_tweets['keyword'].value_counts().index[:20], palette='magma')
plt.title('Top 20 Most Frequent Keywords in Disaster Tweets')
plt.xlabel('Number of Tweets')
plt.ylabel('Keyword')
plt.show()

In [ ]:
# Analyze tweet lengths (number of words)
df_tweets['word_count'] = df_tweets['text'].apply(lambda x: len(str(x).split()))

# Analyze character counts
df_tweets['char_count'] = df_tweets['text'].apply(lambda x: len(str(x)))

print("--- Tweet Length Analysis ---")
print(f"Average word count: {df_tweets['word_count'].mean():.2f}")
print(f"Average character count: {df_tweets['char_count'].mean():.2f}")

# Distribution of word counts
plt.figure(figsize=(12, 5))
sns.histplot(df_tweets['word_count'], bins=50, kde=True, color='skyblue')
plt.title('Distribution of Word Counts in Tweets')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.show()

# Distribution of character counts
plt.figure(figsize=(12, 5))
sns.histplot(df_tweets['char_count'], bins=50, kde=True, color='salmon')
plt.title('Distribution of Character Counts in Tweets')
plt.xlabel('Number of Characters')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Compare word and character counts for disaster vs. non-disaster tweets
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.boxplot(x='target', y='word_count', data=df_tweets, palette='Pastel1')
plt.title('Word Count Distribution by Target')
plt.xlabel('Target (0 = Not Disaster, 1 = Disaster)')
plt.ylabel('Word Count')

plt.subplot(1, 2, 2)
sns.boxplot(x='target', y='char_count', data=df_tweets, palette='Pastel2')
plt.title('Character Count Distribution by Target')
plt.xlabel('Target (0 = Not Disaster, 1 = Disaster)')
plt.ylabel('Character Count')

plt.tight_layout()
plt.show()

print("\n--- Average Word and Character Counts by Target ---")
display(df_tweets.groupby('target')[['word_count', 'char_count']].mean())


In [ ]:
# Analyze the 'location' column
print("\n--- Location Analysis ---")
print(f"Number of unique locations: {df_tweets['location'].nunique()}")
print(f"Number of missing locations: {df_tweets['location'].isnull().sum()} ({(df_tweets['location'].isnull().sum() / len(df_tweets) * 100):.2f}%)")

# Top 10 most frequent locations
plt.figure(figsize=(10, 6))
sns.barplot(x=df_tweets['location'].value_counts().head(10).values, y=df_tweets['location'].value_counts().head(10).index, palette='viridis')
plt.title('Top 10 Most Frequent Locations')
plt.xlabel('Number of Tweets')
plt.ylabel('Location')
plt.show()

### 9.4 Modeling

### 9.5 GloVe - LSTM for Disaster Tweets

We will now apply GloVe embeddings and an LSTM model to the `df_tweets` dataset to classify whether a tweet indicates a real disaster or not. This involves cleaning and stemming the tweet text, converting them into numerical representations using GloVe, and then training a recurrent neural network.

In [ ]:
# Apply cleaning and stemming functions to the 'text' column of df_tweets
# The 'clean_text' and 'stem_text' functions were defined earlier for the Titanic dataset.
df_tweets['Cleaned_Tweet'] = df_tweets['text'].apply(clean_text)
df_tweets['Stemmed_Tweet'] = df_tweets['Cleaned_Tweet'].apply(stem_text)

print("First 10 original tweets vs. cleaned and stemmed tweets:")
for i in range(10):
    print(f"Original: {df_tweets['text'].iloc[i]} -> Cleaned & Stemmed: {df_tweets['Stemmed_Tweet'].iloc[i]}")

In [ ]:
# Generate GloVe embeddings for the stemmed tweets
# The 'glove_model' and 'vector_size' were loaded previously for the Titanic dataset.
df_tweets['GloVe_Embeddings'] = df_tweets['Stemmed_Tweet'].apply(lambda x: get_avg_word_embedding(x, glove_model, vector_size))

print("\nFirst 5 rows of GloVe Embeddings for tweets (average vector for each tweet):")
display(df_tweets['GloVe_Embeddings'].head())

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import numpy as np

# Filter out rows where 'Stemmed_Tweet' might be empty or NaN after processing
tweet_mask = df_tweets['Stemmed_Tweet'].apply(lambda x: len(x) > 0)
tweets_for_lstm = df_tweets['Stemmed_Tweet'][tweet_mask].tolist()
y_tweets_lstm = df_tweets['target'][tweet_mask].values

# Tokenize the words
tokenizer_tweets = Tokenizer(num_words=10000, oov_token="<unk>") # Increased num_words for tweet vocabulary
tokenizer_tweets.fit_on_texts(tweets_for_lstm)

# Convert text to sequences of integers
sequences_tweets = tokenizer_tweets.texts_to_sequences(tweets_for_lstm)

# Determine maximum sequence length for padding
max_sequence_len_tweets = max([len(x) for x in sequences_tweets])
print(f"Maximum sequence length for tweets: {max_sequence_len_tweets}")

# Pad sequences to ensure uniform input size
X_tweets_lstm = pad_sequences(sequences_tweets, maxlen=max_sequence_len_tweets, padding='post')

print(f"Shape of X_tweets_lstm (padded sequences): {X_tweets_lstm.shape}")
print(f"Shape of y_tweets_lstm: {y_tweets_lstm.shape}")

# Split data into training and testing sets for LSTM
X_train_tweets, X_test_tweets, y_train_tweets, y_test_tweets = train_test_split(
    X_tweets_lstm, y_tweets_lstm, test_size=0.2, random_state=42, stratify=y_tweets_lstm
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

# Define vocabulary size for tweets
vocab_size_tweets = len(tokenizer_tweets.word_index) + 1

# Build the LSTM model for tweets
embedding_dim_tweets = 100 # Consistent with GloVe vector size if pre-trained embeddings were used, or can be tuned
model_tweets = Sequential([
    Embedding(vocab_size_tweets, embedding_dim_tweets),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Output layer for binary classification
])

model_tweets.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print(model_tweets.summary())

# Train the model
early_stopping_tweets = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_tweets = model_tweets.fit(
    X_train_tweets, y_train_tweets,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping_tweets]
)

# Evaluate the model on the test set
loss_tweets, accuracy_tweets = model_tweets.evaluate(X_test_tweets, y_test_tweets, verbose=0)
print(f"\nLSTM Model Test Accuracy (Tweets): {accuracy_tweets:.4f}")

# Make predictions for classification report and confusion matrix
y_pred_tweets_prob = model_tweets.predict(X_test_tweets)
y_pred_tweets = (y_pred_tweets_prob > 0.5).astype(int)

print("\nClassification Report (LSTM - Tweets):\n")
print(classification_report(y_test_tweets, y_pred_tweets))

print("\nConfusion Matrix (LSTM - Tweets):\n")
print(confusion_matrix(y_test_tweets, y_pred_tweets))